# Norway vs Brazil R16 prediction

Round of 16 prediction for Norway vs Brazil on 2026-07-05.

In [7]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import poisson

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import load_data as L
from src.features import build_features as F
from src.models import logistic_model as LM
from src.models import poisson_model as P
from src.utils.config import RESULT_CLASSES

TEAM = "Norway"
OPPONENT = "Brazil"
MATCH_DATE = "2026-07-05"
FIXTURE = [{"date": MATCH_DATE, "team": TEAM, "opponent": OPPONENT, "is_home": 1, "is_neutral": 1}]


## Latest Brazil evidence

The local `matches.csv` already includes Brazil through the 2026-06-13 World Cup draw with Morocco. The supplied Flashscore run adds Brazil's remaining pre-R16 competitive results in memory before feature engineering.


In [8]:
BRAZIL_RESULT_UPDATES = pd.DataFrame(
    [
        {
            "match_id": "2026-06-20_BR_HT_flashscore",
            "date": "2026-06-20",
            "home_team": "Brazil",
            "away_team": "Haiti",
            "home_score": 3,
            "away_score": 0,
            "competition": "FIFA World Cup",
            "competition_type": "world_cup",
            "neutral": True,
            "source": "flashscore.com",
        },
        {
            "match_id": "2026-06-24_SQ_BR_flashscore",
            "date": "2026-06-24",
            "home_team": "Scotland",
            "away_team": "Brazil",
            "home_score": 0,
            "away_score": 3,
            "competition": "FIFA World Cup",
            "competition_type": "world_cup",
            "neutral": True,
            "source": "flashscore.com",
        },
        {
            "match_id": "2026-06-29_BR_JP_flashscore",
            "date": "2026-06-29",
            "home_team": "Brazil",
            "away_team": "Japan",
            "home_score": 2,
            "away_score": 1,
            "competition": "FIFA World Cup",
            "competition_type": "world_cup",
            "neutral": True,
            "source": "flashscore.com",
        },
    ]
)
BRAZIL_RESULT_UPDATES["date"] = pd.to_datetime(BRAZIL_RESULT_UPDATES["date"])

flashscore_sources = pd.DataFrame(
    [
        ("2026-06-29", "Brazil 2-1 Japan", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/japan-ULXPdOUj/summary/stats/overall/?mid=f7ENGzPc"),
        ("2026-06-24", "Scotland 0-3 Brazil", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/scotland-fZRU25WH/summary/stats/overall/?mid=EgVZxtj1"),
        ("2026-06-20", "Brazil 3-0 Haiti", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/haiti-nk4v10Z1/summary/stats/overall/?mid=IRyRv2Ll"),
        ("2026-06-13", "Brazil 1-1 Morocco", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/morocco-IDKYO3R8/summary/stats/overall/?mid=b5JayTEd"),
        ("2026-06-06", "Brazil 2-1 Egypt", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/egypt-bejDn7NN/summary/stats/overall/?mid=r3s2pV84"),
        ("2026-05-31", "Brazil 6-2 Panama", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/panama-OWKqbCfi/summary/stats/overall/?mid=4UXzDy1d"),
        ("2026-03-31", "Brazil 3-1 Croatia", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/croatia-K8aznggo/summary/stats/overall/?mid=A3LetXFE"),
        ("2026-03-26", "France 2-1 Brazil", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/france-QkGeVG1n/summary/stats/overall/?mid=few9ITN7"),
        ("2025-11-18", "Brazil 1-1 Tunisia", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/tunisia-QqZVYk95/summary/stats/overall/?mid=WbBoTNBE"),
        ("2025-11-15", "Brazil 2-0 Senegal", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/senegal-hOIsJLJr/summary/stats/overall/?mid=AeBovozo"),
        ("2025-10-14", "Japan 3-2 Brazil", "https://www.flashscore.com/match/football/brazil-I9l9aqLq/japan-ULXPdOUj/summary/stats/overall/?mid=l4iAKLHs"),
    ],
    columns=["date", "match", "stats_url"],
)

display(flashscore_sources)


,date,match,stats_url
0,2026-06-29,Brazil 2-1 Japan,https://www.flashscore.com/match/football/braz...
1,2026-06-24,Scotland 0-3 Brazil,https://www.flashscore.com/match/football/braz...
2,2026-06-20,Brazil 3-0 Haiti,https://www.flashscore.com/match/football/braz...
3,2026-06-13,Brazil 1-1 Morocco,https://www.flashscore.com/match/football/braz...
4,2026-06-06,Brazil 2-1 Egypt,https://www.flashscore.com/match/football/braz...
5,2026-05-31,Brazil 6-2 Panama,https://www.flashscore.com/match/football/braz...
6,2026-03-31,Brazil 3-1 Croatia,https://www.flashscore.com/match/football/braz...
7,2026-03-26,France 2-1 Brazil,https://www.flashscore.com/match/football/braz...
8,2025-11-18,Brazil 1-1 Tunisia,https://www.flashscore.com/match/football/braz...
9,2025-11-15,Brazil 2-0 Senegal,https://www.flashscore.com/match/football/braz...


## Build fixture dataset

The pre-match Elo override is already present locally as of 2026-07-04: Norway 1934 and Brazil 2031. Because rating joins use strictly prior values, the 2026-07-05 fixture picks up those pre-match ratings.


In [9]:
matches_base = L.load_matches()
elo = L.load_elo()
fifa = L.load_fifa()
flashscore = L.load_flashscore()

matches = matches_base[~matches_base["match_id"].isin(BRAZIL_RESULT_UPDATES["match_id"])].copy()
matches = (
    pd.concat([matches, BRAZIL_RESULT_UPDATES], ignore_index=True)
    .sort_values(["date", "match_id"])
    .reset_index(drop=True)
)

persp_feat, model_ds, all_ds, fixture_ds = F.engineer(
    matches, elo, fifa, flashscore, fixtures=FIXTURE
)

fixture_row = fixture_ds[(fixture_ds["team"] == TEAM) & (fixture_ds["opponent"] == OPPONENT)].iloc[0]

print(f"base matches: {len(matches_base):,}  |  with Brazil updates: {len(matches):,}")
print(f"Flashscore rows in canonical input: {len(flashscore):,}")
print(f"fixture: {TEAM} vs {OPPONENT} on {MATCH_DATE} (neutral)")
print(f"Elo before fixture: {TEAM} {fixture_row['team_elo_before']:.0f}, {OPPONENT} {fixture_row['opponent_elo_before']:.0f}")
print(f"Elo gap: {fixture_row['elo_diff']:+.0f}")


19:51:06 | INFO    | src.preprocessing.load_data | loaded 4647 matches (2020-12-04 -> 2026-06-30)
19:51:07 | INFO    | src.preprocessing.load_data | loaded 516 Flashscore team-match rows (107 teams)
19:51:07 | INFO    | src.preprocessing.load_data | team-perspective rows: 9300 (4650 matches x 2 sides)
19:51:07 | INFO    | src.preprocessing.load_data | attached Flashscore stats (xg coverage: 193/9300 perspective rows)
19:51:22 | INFO    | src.features.build_features | model dataset: 296 focus rows | 9308 all-team rows | 1 fixture rows
base matches: 4,647  |  with Brazil updates: 4,650
Flashscore rows in canonical input: 516
fixture: Norway vs Brazil on 2026-07-05 (neutral)
Elo before fixture: Norway 1934, Brazil 2031
Elo gap: -97


## Prediction

The ensemble is the same simple average used elsewhere in the project: multinomial logistic W/D/L probabilities plus independent-Poisson score probabilities.


In [10]:
fit = LM.fit_final(model_ds, train_override=all_ds)
log_model, features = fit["logistic"]

log_p = LM.proba_frame(log_model, fixture_ds.loc[[fixture_row.name], features]).iloc[0].to_dict()

mu, strengths = P.compute_strengths(persp_feat)
pois = P.predict(
    strengths,
    mu,
    fixture_row["team"],
    fixture_row["opponent"],
    int(fixture_row["is_home"]),
    int(fixture_row["is_neutral"]),
)
ens = {c: (log_p[c] + pois[c]) / 2 for c in RESULT_CLASSES}

summary = pd.DataFrame(
    {
        "Norway win": [log_p["win"], pois["win"], ens["win"]],
        "Draw": [log_p["draw"], pois["draw"], ens["draw"]],
        "Brazil win": [log_p["loss"], pois["loss"], ens["loss"]],
    },
    index=["Logistic", "Poisson", "Ensemble"],
)

display((summary * 100).round(1).astype(str) + "%")

verdict = {"win": "Norway", "draw": "Draw after 90 minutes", "loss": OPPONENT}[max(ens, key=ens.get)]
advance_norway = ens["win"] + 0.5 * ens["draw"]
advance_brazil = ens["loss"] + 0.5 * ens["draw"]

print(f"90-minute ensemble lean: {verdict}")
print(f"Expected goals: Norway {pois['lam_team']:.2f} | Brazil {pois['lam_opp']:.2f}")
print(f"Most likely Poisson scoreline: Norway {pois['score'][0]}-{pois['score'][1]} Brazil")
print(f"Naive advancement estimate if draw is split evenly: Norway {advance_norway:.1%}, Brazil {advance_brazil:.1%}")


19:51:23 | INFO    | src.models.logistic_model | fitted logistic on 9308 rows (half-life=1095 d)
19:51:23 | INFO    | src.models.poisson_model | Poisson strengths: 201 teams, league avg = 1.32 goals/team/match


c:\Users\karoo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


,Norway win,Draw,Brazil win
Logistic,18.0%,27.9%,54.1%
Poisson,33.3%,24.7%,42.0%
Ensemble,25.7%,26.3%,48.0%


90-minute ensemble lean: Brazil
Expected goals: Norway 1.34 | Brazil 1.54
Most likely Poisson scoreline: Norway 1-1 Brazil
Naive advancement estimate if draw is split evenly: Norway 38.8%, Brazil 61.2%


## Why Brazil leads

Brazil's pre-match Elo edge is about 100 points and the injected World Cup wins improve their recent goal-difference form. Norway still has a path in the Poisson score model, but the logistic model reacts strongly to the Elo and form gap.


In [11]:
feature_snapshot = pd.Series(
    {
        "elo_diff": fixture_row["elo_diff"],
        "rest_days_diff": fixture_row["rest_days_diff"],
        "win_rate_last_5_diff": fixture_row["win_rate_last_5_diff"],
        "points_per_game_last_5_diff": fixture_row["points_per_game_last_5_diff"],
        "goal_diff_avg_last_5_diff": fixture_row["goal_diff_avg_last_5_diff"],
        "goal_diff_avg_last_10_diff": fixture_row["goal_diff_avg_last_10_diff"],
        "Norway win_rate_last_5": fixture_row["team_win_rate_last_5"],
        "Brazil win_rate_last_5": fixture_row["opponent_win_rate_last_5"],
        "Norway ppg_last_5": fixture_row["team_points_per_game_last_5"],
        "Brazil ppg_last_5": fixture_row["opponent_points_per_game_last_5"],
        "Norway gd_last_5": fixture_row["team_goal_diff_avg_last_5"],
        "Brazil gd_last_5": fixture_row["opponent_goal_diff_avg_last_5"],
    },
    name="value",
)
display(feature_snapshot.to_frame().round(3))

latest = matches[(matches["home_team"].isin([TEAM, OPPONENT])) | (matches["away_team"].isin([TEAM, OPPONENT]))]
latest = latest.sort_values("date")[["date", "home_team", "away_team", "home_score", "away_score", "competition_type", "source"]]
display(latest.tail(14))


,value
elo_diff,-97.000
rest_days_diff,-1.000
win_rate_last_5_diff,-0.059
points_per_game_last_5_diff,-0.353
goal_diff_avg_last_5_diff,-1.235
goal_diff_avg_last_10_diff,-0.761
Norway win_rate_last_5,0.706
Brazil win_rate_last_5,0.765
Norway ppg_last_5,2.176
Brazil ppg_last_5,2.529


,date,home_team,away_team,home_score,away_score,competition_type,source
4455,2026-03-31,Brazil,Croatia,3,1,friendly,eloratings.net
4482,2026-03-31,Norway,Switzerland,0,0,friendly,eloratings.net
4515,2026-05-31,Brazil,Panama,6,2,friendly,eloratings.net
4527,2026-06-01,Norway,Sweden,3,1,friendly,eloratings.net
4578,2026-06-06,Brazil,Egypt,2,1,friendly,eloratings.net
4601,2026-06-07,Morocco,Norway,1,1,friendly,eloratings.net
4635,2026-06-13,Brazil,Morocco,1,1,world_cup,eloratings.net
4640,2026-06-16,Iraq,Norway,1,4,world_cup,eloratings.net
4641,2026-06-20,Brazil,Haiti,3,0,world_cup,flashscore.com
4644,2026-06-23,Norway,Senegal,3,2,world_cup,eloratings.net


## Scoreline grid

Independent Poisson scoreline probabilities, shown as a 0-6 grid plus the top five exact scores.


In [12]:
MAXG = 6
goals = np.arange(MAXG + 1)
a = poisson.pmf(goals, pois["lam_team"])
b = poisson.pmf(goals, pois["lam_opp"])
grid = np.outer(a, b)
grid = grid / grid.sum()

score_grid = pd.DataFrame(
    grid,
    index=pd.Index(range(MAXG + 1), name="Norway"),
    columns=pd.Index(range(MAXG + 1), name="Brazil"),
)
display((score_grid * 100).round(2))

flat = []
for i in range(MAXG + 1):
    for j in range(MAXG + 1):
        flat.append((f"{i}-{j}", grid[i, j]))
top5 = pd.DataFrame(flat, columns=["score", "probability"]).sort_values("probability", ascending=False).head(5)
display(top5.assign(probability=(top5["probability"] * 100).round(2).astype(str) + "%"))


Brazil,0,1,2,3,4,5,6
Norway,,,,,,,
0,5.62,8.63,6.64,3.40,1.31,0.40,0.10
1,7.55,11.60,8.92,4.57,1.76,0.54,0.14
2,5.07,7.80,5.99,3.07,1.18,0.36,0.09
3,2.27,3.49,2.68,1.38,0.53,0.16,0.04
4,0.76,1.17,0.90,0.46,0.18,0.05,0.01
5,0.21,0.32,0.24,0.12,0.05,0.01,0.00
6,0.05,0.07,0.05,0.03,0.01,0.00,0.00


,score,probability
8,1-1,11.6%
9,1-2,8.92%
1,0-1,8.63%
15,2-1,7.8%
7,1-0,7.55%


## Current read

Ensemble leans Brazil over 90 minutes. Brazil about 48%, draw about 26%, Norway about 26%. The most likely Poisson scoreline is 1-1, which keeps the knockout path live, but a simple split of the draw bucket gives a rough advancement estimate of Brazil about 61% vs Norway about 39%.
